## Load stimuli saved with QDSpy

Requires the `.pickle` files saved with QDSpy, which are not in the repository because of size.  

TODOs: 
- Make these files available somewhere else
- Get exact movie scaling parameters from `QDSpy.ini` file
- Generate movies with a centre pixel (e.g., 41x41)

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

HERE = os.getcwd()
sys.path.append(HERE)
sys.path.append(os.path.join(HERE, "..", "..", "utils"))
from data_io import get_data_config, REPO_ROOT
from stim_utils.stimulus import stim_movies


In [11]:
'''
# Full QDSpy movie-as-pickle files are not part of the shared eyewire2-data download
# (only much smaller per-stimulus pickles are) -- place them here manually if needed.
STIM_MOV_PATH = Path(HERE) / "stimuli-as-movies"
STIM_MOV_EXT = ".pickle"
'''
# data_config.yaml's paths are written relative to a notebook directly under
# notebooks/ (one level under the repo root); anchor there before applying it,
# since this script sits one level deeper, under notebooks/light_exposure/.
DATA_2P = (Path(REPO_ROOT) / "notebooks" / get_data_config()["data_2p_dir"]).resolve()

STIM_MOV_EXT = ".pickle"
STIM_MOV_PATH = DATA_2P / "stimuli" / "QDSpy"
STIM_DS = "DS" # RGC_MovingBar
STIM_CHIRP = "Chirp" # RGC_Chirp
STIM_MC_LEFT = "MouseCam_Left"

FIG_DIR = os.path.join(HERE, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

### Load stimulus movie files into numpy array

... and zero blue channel, as it was not used (green=G, red=UV?)

In [12]:
# Load movie files and zero blue channel
#tmp_path = Path.joinpath(STIM_MOV_PATH, "RGC_MovingBar" +STIM_MOV_EXT)
tmp_path = Path.joinpath(STIM_MOV_PATH, STIM_DS +STIM_MOV_EXT)
mov_DS = stim_movies.load_qdspy_movie(tmp_path)
mov_DS[:,:,:,2] = 0

tmp_path = Path.joinpath(STIM_MOV_PATH, STIM_CHIRP +STIM_MOV_EXT)
mov_Chirp = stim_movies.load_qdspy_movie(tmp_path)
mov_Chirp[:,:,:,2] = 0

tmp_path = Path.joinpath(STIM_MOV_PATH, STIM_MC_LEFT +STIM_MOV_EXT)
mov_MouseCamLeft = stim_movies.load_qdspy_movie(tmp_path)
mov_MouseCamLeft[:,:,:,2] = 0

# Define spatial and temporal scaling (approximated, see TODOs)
# (pixel size from moving bar width / bar pixels in movies)
_, dx, dy, _ = mov_DS.shape
px_um = 300 /7
params = dict({
    "pix_size_um": 300 /7,  # moving bar width / bar pixels in movies
    "mov_dxy": [dx, dy],
    "mov_dxy_um": [px_um *dx, px_um *dy],
    "dt_fr_s": 1 /60,
    "nCh": 2
})

Loading pickle file from: F:\Git\huggingface\eyewire2-data\data-2p\stimuli\QDSpy\DS.pickle
Error: File does not contain a simple numpy array as expected.


TypeError: 'NoneType' object does not support item assignment

### Inspect movies, if needed

In [ ]:
_mov = mov_DS

nCh = 3
nFr = _mov.shape[0]
vmin = _mov.min()
vmax = _mov.max()

def show_frame(frame):
    fig, axes = plt.subplots(1, nCh, figsize=(12, 3))
    for ch in range(nCh):
        axes[ch].imshow(_mov[frame,:,:,ch], cmap='gray', vmin=vmin, vmax=vmax)
        axes[ch].axis('off')
        axes[ch].set_title(f'Channel {ch}')
    fig.suptitle(f'Frame {frame}/{nFr-1}')
    plt.tight_layout()
    plt.show()

# Create interactive slider
interact(show_frame, frame=IntSlider(min=0, max=nFr-1, step=1, value=0, description='Frame:'))

### Calculate intensity traces for an area within the movie

In [ ]:
DS_intens, DS_intens_cumul = stim_movies.calc_intensity_trace(
    mov_DS, params, _range_s=[0, -1], _field_xy_um=[0,0], _field_size_um=[95*2.0, 95*2.0],
    _plot=True, _verbose=True
)
params

### Flattening movies 

In [ ]:
mov_flat = stim_movies.flatten_movie(mov_DS, params, _range_s=[0, 25], _plot=True)